<a href="https://colab.research.google.com/github/aditya-ailsinghani/fifa22-player-value-analytics/blob/main/modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
PROJECT_PATH = "/content/drive/MyDrive/Projects/Fifa_Analytics"
os.chdir(PROJECT_PATH)

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [4]:
df = pd.read_csv("FIFA22_cleaned.csv")
df.shape

(16710, 64)

In [5]:
df["log_value"] = np.log1p(df["Value_EUR"])

In [6]:
df[["Value_EUR", "log_value"]].head()

,Value_EUR,log_value
0,107500000.0,18.493001
1,93000000.0,18.348110
2,44500000.0,17.611000
3,125500000.0,18.647816
4,37000000.0,17.426428


The features used for modeling were selected based on insights from the exploratory data analysis rather than chosen arbitrarily. The EDA showed that player market value is most strongly differentiated by technical and mental attributes, with skills such as ball control, passing, dribbling, vision, finishing, and composure consistently separating high-value players from the rest. These attributes exhibited the largest gaps between top- and lower-value players both overall and within specific positional roles.

In addition, age was included as a contextual variable, as player value was observed to peak within a limited age range and decline beyond a player’s prime years. Overall and potential ratings were incorporated to capture a player’s current performance level and future upside, which are central to real-world valuation. Finally, selected physical attributes were included to account for supporting athletic factors, though EDA indicated that these played a secondary role compared to technical and mental skills. This feature set balances interpretability and predictive power while remaining directly grounded in observed patterns from the data.

In [7]:
feature_cols = [
    # Demographics
    "Age",

    # Core ratings
    "Overall",
    "Potential",

    # Technical attributes
    "Finishing",
    "Dribbling",
    "ShortPassing",
    "BallControl",
    "Vision",
    "Composure",

    # Physical attributes
    "Acceleration",
    "SprintSpeed",
    "Stamina",
    "Strength"
]

In [8]:
X = df[feature_cols]
y = df["log_value"]

X.shape, y.shape

((16710, 13), (16710,))

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((13368, 13), (3342, 13))

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [11]:
y_pred = lr.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2, rmse

(0.2543892501960242, np.float64(1.9732540517487978))

The model is better at ranking and approximating player value than predicting exact prices.`

In [12]:
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr.coef_
}).sort_values(by="Coefficient", ascending=False)

coef_df

,Feature,Coefficient
1,Overall,0.244838
4,Dribbling,0.012288
8,Composure,0.010939
7,Vision,0.005485
11,Stamina,0.004479
10,SprintSpeed,0.002186
5,ShortPassing,0.001821
12,Strength,0.001696
3,Finishing,0.001050
9,Acceleration,-0.009535


Baseline Model(Without Overall)

In [13]:
feature_cols_no_overall = [
    "Age",
    "Potential",
    "Finishing",
    "Dribbling",
    "ShortPassing",
    "BallControl",
    "Vision",
    "Composure",
    "Acceleration",
    "SprintSpeed",
    "Stamina",
    "Strength"
]

X2 = df[feature_cols_no_overall]
y = df["log_value"]

In [14]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

lr2 = LinearRegression()
lr2.fit(X2_train, y2_train)

y2_pred = lr2.predict(X2_test)

r2_2 = r2_score(y2_test, y2_pred)
rmse_2 = np.sqrt(mean_squared_error(y2_test, y2_pred))

r2_2, rmse_2

(0.20673415790316174, np.float64(2.03533686979248))

To better understand the contribution of individual player attributes, a second linear regression model was trained after removing the overall rating. The motivation for this experiment was to reduce reliance on FIFA’s summary score and instead evaluate how well specific technical, mental, and physical attributes explain player market value on their own. This model is therefore less focused on predictive performance and more focused on interpretability.

As expected, removing the overall rating led to a modest decrease in model performance, with the explained variance dropping slightly compared to the baseline model that included overall rating. This indicates that the overall rating captures a substantial amount of information relevant to player valuation. However, the model still explains a meaningful portion of market value, confirming that individual performance attributes contain independent valuation signal. This result supports the exploratory analysis, which showed that technical and mental skills play an important role in differentiating high- and low-value players, even when a summary performance metric is excluded.

Non Linear Model(Random Forest)

In [15]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [16]:
rf_pred = rf.predict(X_test)

r2_rf = r2_score(y_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))

r2_rf, rmse_rf

(0.3006723267358036, np.float64(1.9110290103267902))

To capture non-linear relationships and interactions between player attributes, a Random Forest regression model was trained using the same feature set as the baseline model. Unlike linear regression, the Random Forest can model threshold effects and complex interactions, which are expected in player valuation where small improvements at elite levels can lead to disproportionately large increases in market value.

The Random Forest model achieved improved performance compared to the linear baselines, indicating that non-linear patterns play an important role in explaining player market value. While the model still does not capture all sources of variation—such as league context, contract details, or off-field factors—it provides a stronger predictive baseline and confirms that performance attributes contain substantial, but not complete, valuation signal.

Feature Importance

In [17]:
importances = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

importances

,Feature,Importance
2,Potential,0.175367
0,Age,0.131764
1,Overall,0.112357
8,Composure,0.076611
3,Finishing,0.069563
4,Dribbling,0.058155
12,Strength,0.058090
9,Acceleration,0.055786
7,Vision,0.055453
11,Stamina,0.054838


Feature importance from the Random Forest model highlights the relative influence of different attributes in estimating player market value. Potential and age emerge as the most influential factors, reflecting the market’s strong emphasis on future upside and career stage. While the overall rating remains an important input, it does not dominate the model, indicating that player valuation depends on a combination of summary performance, future potential, and individual attributes.

Technical and mental attributes such as composure, finishing, dribbling, vision, and ball control consistently contribute to the model’s predictions, reinforcing findings from the exploratory analysis that decision-making and technical quality are key drivers of value. Physical attributes such as speed, stamina, and strength also play a role, but their relative importance is lower, suggesting they act as supporting rather than defining factors. Overall, the feature importance results align closely with earlier EDA insights and provide a coherent, data-driven explanation of player valuation.

**Takeaway**

This modeling exercise evaluated multiple approaches to estimating FIFA 22 player market value using performance attributes and demographics. A linear regression model provided a transparent baseline, demonstrating that player attributes explain a meaningful portion of market value, though relationships are limited by linear assumptions. Removing the overall rating reduced predictive performance but offered clearer insight into the role of individual skills.

A Random Forest regression model improved predictive accuracy by capturing non-linear relationships and interactions, confirming that player valuation is influenced by complex effects involving age, potential, and performance attributes. Across all models, future potential, age, technical quality, and mental attributes consistently emerged as key drivers of value, while physical traits played a secondary role.

While the models do not capture all real-world factors affecting player valuation—such as league context, contract details, or off-field considerations, they provide a realistic and interpretable framework for understanding how on-field performance translates into market value. This analysis demonstrates how exploratory insights can be systematically translated into predictive models while maintaining interpretability and business relevance.